# HCDE 530 — Assignment 5 (Week 5): Pandas — The Met Open Access (MP1)

**Dataset:** artwork metadata from **The Metropolitan Museum of Art Collection API** ([documentation](https://metmuseum.github.io/)), saved as `artworks.csv` in this folder.

**Important — scope:** This file is **not** the full Met collection; it is a **small extract of 50 objects** for this assignment / MP1. All results apply to **this sample only**.

## Three analytical questions

1. **How are artworks distributed across different departments?**
2. **Are there missing values in key fields such as date, medium, or classification?**
3. **How does artwork medium vary across different departments?**

The notebook uses pandas operations from class (`head` / `info`, `value_counts`, filtering, `groupby` + `mean`, `isnull`) and **plain-English `#` comments** under each question explaining what is asked and what the output means.

In [1]:
from pathlib import Path
from typing import Tuple

import pandas as pd


def load_met_artworks_csv() -> Tuple[pd.DataFrame, Path]:
    """Load MP1 Met artworks CSV from this folder or the shared week4 export."""
    here = Path(".").resolve()
    search = [
        here / "artworks.csv",
        here.parent / "week4" / "met_api_a4" / "artworks.csv",
    ]
    for p in search:
        if p.is_file():
            return pd.read_csv(p), p
    raise FileNotFoundError(
        "Could not find artworks.csv. Place it next to this notebook or under week4/met_api_a4/."
    )


df, csv_path = load_met_artworks_csv()
print("Loaded:", csv_path)
print("Shape (rows, columns):", df.shape)
list(df.columns)

Loaded: /Users/hzyyy/Desktop/HCDE530-files/A5(week5 hw)/artworks.csv
Shape (rows, columns): (50, 8)


['objectID',
 'title',
 'artistDisplayName',
 'objectDate',
 'department',
 'medium',
 'primaryImageSmall',
 'objectURL']

---
## Setup — What does the table look like? (`head`, `info`)

In [2]:
# I am asking: what columns and example rows do I have before answering the three questions?
# The answer grounds me in field names (e.g. objectDate, department, medium) and typical text length.
df.head()

,objectID,title,artistDisplayName,objectDate,department,medium,primaryImageSmall,objectURL
0,39092,Bodhisattva Avaolkiteshvara,NaN,second half of the 7th–early 8th century,Asian Art,Copper alloy,https://images.metmuseum.org/CRDImages/as/web-...,https://www.metmuseum.org/art/collection/searc...
1,310552,Incense burner with feline head,Tiwanaku artist(s),500–1000 CE,The Michael C. Rockefeller Wing,"Ceramic, slip",https://images.metmuseum.org/CRDImages/ao/web-...,https://www.metmuseum.org/art/collection/searc...
2,570719,Cat with image of Bastet on breast,NaN,664–30 B.C.,Egyptian Art,Cupreous metal,https://images.metmuseum.org/CRDImages/eg/web-...,https://www.metmuseum.org/art/collection/searc...
3,436545,Manuel Osorio Manrique de Zuñiga (1784–1792),Goya (Francisco de Goya y Lucientes),1787–88,European Paintings,Oil on canvas,https://images.metmuseum.org/CRDImages/ep/web-...,https://www.metmuseum.org/art/collection/searc...
4,551786,"Book of the Dead of the Priest of Horus, Imhot...",NaN,ca. 332–200 B.C.,Egyptian Art,"Papyrus, ink",https://images.metmuseum.org/CRDImages/eg/web-...,https://www.metmuseum.org/art/collection/searc...


In [3]:
# I am asking: how many rows/columns are there and what dtypes and non-null counts does pandas see?
# The answer sets expectations for how reliable counts and missing-value checks will be in a 50-row extract.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   objectID           50 non-null     int64 
 1   title              50 non-null     object
 2   artistDisplayName  37 non-null     object
 3   objectDate         49 non-null     object
 4   department         50 non-null     object
 5   medium             50 non-null     object
 6   primaryImageSmall  48 non-null     object
 7   objectURL          50 non-null     object
dtypes: int64(1), object(7)
memory usage: 3.2+ KB


---
## Question 1 — How are artworks distributed across different departments?

Use **`value_counts()`** to see counts (and optional proportions) per `department`.

In [4]:
# I am asking: in this sample, how many objects fall into each curatorial department?
# The answer shows imbalance (e.g. more European Paintings than Islamic Art) — important when comparing departments in MP1.
counts = df["department"].value_counts()
props = df["department"].value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({"count": counts, "percent_of_sample": props})

,count,percent_of_sample
department,,
European Paintings,14,28.0
Asian Art,9,18.0
European Sculpture and Decorative Arts,9,18.0
The Michael C. Rockefeller Wing,5,10.0
Egyptian Art,3,6.0
Robert Lehman Collection,3,6.0
Islamic Art,3,6.0
Drawings and Prints,2,4.0
Medieval Art,1,2.0


---
## Question 2 — Are there missing values in key fields (date, medium, classification)?

This CSV includes **`objectDate`** (date text from the API) and **`medium`**. The full Met object JSON can include a **`classification`** field, but **this export does not have a `classification` column** — we still check **`department`** as the stable curatorial label alongside date and medium.

Use **`isnull().sum()`** on those fields, then **filter** rows with gaps to inspect examples.

In [5]:
# I am asking: among objectDate, medium, and department, how many cells are missing per column?
# The answer tells me whether I can treat dates/media as complete for every row or need imputation or drops in MP1.
key_cols = ["objectDate", "medium", "department"]
df[key_cols].isnull().sum()

objectDate    1
medium        0
department    0
dtype: int64

In [6]:
# I am asking: which rows still lack a date or medium string so I can read them qualitatively?
# The answer lists concrete records (sample IDs/titles) instead of only aggregate counts.
missing_date_or_medium = df[df["objectDate"].isna() | df["medium"].isna()]
print(f"Rows missing objectDate or medium: {len(missing_date_or_medium)}")
missing_date_or_medium[["objectID", "title", "department", "objectDate", "medium"]]

Rows missing objectDate or medium: 1


,objectID,title,department,objectDate,medium
45,436885,Scene from a Novella,European Paintings,NaN,Tempera on wood


---
## Question 3 — How does artwork medium vary across different departments?

Describe variation with **counts of distinct media per department** and **`groupby` + `mean`** on a simple numeric summary (average length of the medium string), which captures how verbose material descriptions are by department in this extract.

In [7]:
# I am asking: within each department, how many different medium strings appear (breadth of materials)?
# The answer shows whether a department row is one repeated material (low nunique) or many different media (high nunique).
df.groupby("department")["medium"].nunique().sort_values(ascending=False)

department
Asian Art                                 9
European Paintings                        9
European Sculpture and Decorative Arts    8
The Michael C. Rockefeller Wing           4
Egyptian Art                              3
Islamic Art                               3
Robert Lehman Collection                  3
Drawings and Prints                       2
Ancient West Asian Art                    1
Medieval Art                              1
Name: medium, dtype: int64

In [8]:
# I am asking: on average, how long is the medium text per department (characters)?
# The answer is one numeric summary of how detailed material descriptions are by department; long entries often mix many materials.
med_len = df["medium"].fillna("").str.len()
df.assign(_medium_len=med_len).groupby("department")["_medium_len"].mean().sort_values(ascending=False).round(1)

department
Islamic Art                               72.7
European Sculpture and Decorative Arts    66.2
Medieval Art                              59.0
Drawings and Prints                       51.5
Robert Lehman Collection                  50.0
Asian Art                                 40.0
The Michael C. Rockefeller Wing           23.4
European Paintings                        17.9
Egyptian Art                              14.0
Ancient West Asian Art                    12.0
Name: _medium_len, dtype: float64